## sLTN Tutorial Series

This is the first of five tutorials introducing **sLTN** (Structural Logic Tensor Networks):

1. **Signature & Interpretation** (this notebook) — the vocabulary layer
2. [FOL Formulas](2-FOLFormulas.ipynb) — connectives and quantifiers
3. [Fuzzy Operators & Gradients](3-FuzzyOperators&Gradients.ipynb) — choosing operators for learning
4. [KB & Learning](4-KB&Learning.ipynb) — knowledge bases and training loops
5. [Structural LTN](5-StructuralLTN.ipynb) — reasoning over time, sequences, and graphs

Before running any of them, install the notebook dependencies:

```bash
pip install -e ".[notebooks]"
```

# Tutorial 1 — The Vocabulary: Sorts, Symbols, and Interpretation

sLTN separates **syntax** from **semantics**:

- a `Signature` declares the vocabulary of a logical language: the names of the symbols and their sorts. It contains no tensors and no networks;
- an `Interpretation` gives each symbol a concrete meaning (its *grounding*): a tensor for an individual, a neural network or a Python function for a predicate.

Because of this split, a formula is written and checked once, against the signature, and can then be evaluated with any interpretation: other data, another network, a trained or an untrained model.

| Symbol | Declares | Typical grounding |
|---|---|---|
| Sort | A domain of individuals | A `Type`: the shape of one individual |
| Constant | One named individual | A tensor, or an `nn.Parameter` to learn it |
| Variable | A placeholder for individuals of a sort | A batch of individuals (a tensor) |
| Predicate | A map from individuals to a truth degree in [0, 1] | A neural module or a callable |
| Function | A map from individuals to an individual | A neural module or a callable |

The notebook follows the same split:

1. **Signature**: declare the vocabulary and parse formulas, with no values at all;
2. **Interpretation**: ground every symbol;
3. **Evaluation**: compute truth degrees, and see how the result's axes follow the formula's variables.

In [ ]:
import torch
import torch.nn as nn
import sltn
from sltn import Signature, Interpretation, Type

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## Part 1 — The Signature

### 1.1 Sorts

A **sort** is a domain of individuals: animals, people, images, embeddings. Every other symbol is declared with respect to sorts: a constant or a variable belongs to a sort, a predicate takes arguments of given sorts, a function maps input sorts to an output sort.

Here we declare two sorts: `Animal`, and `Embedding` for a learned representation of an animal.

In [ ]:
sig = Signature("tutorial")

sig.sort("Animal")
sig.sort("Embedding");

### 1.2 Constants and variables

A **constant** names one fixed individual, such as `Fido`. A **variable** stands for individuals of a sort. It is grounded with a finite batch of individuals, and a formula with a free variable is evaluated on each of them.

In [ ]:
sig.constant("Fido", "Animal")
sig.constant("Rex", "Animal")   # we will learn Rex's features

sig.variable("x", "Animal")
sig.variable("y", "Animal");

### 1.3 Predicates and functions

A **predicate** is declared with its name and the sorts of its arguments. A **function** also has an output sort.

In [ ]:
sig.predicate("IsDog", ["Animal"])
sig.predicate("Near", ["Animal"])               # close to a reference point
sig.predicate("Similar", ["Animal", "Animal"])

sig.function("embed", ["Animal"], "Embedding")
sig.function("diff", ["Animal", "Animal"], "Animal")

print(sig)

`Bool` and `Real` are built-in sorts, present in every signature: `Bool` is the sort of truth degrees, `Real` the sort of a single real number.

### 1.4 Parsing formulas: no values needed

The signature is enough to parse a formula. Parsing resolves every name against the signature and builds a formula tree. Nothing has been grounded yet: no tensor, no network.

In [ ]:
phi = sig.parse("Similar(x, Fido)")
print(phi, "   ->", type(phi).__name__)

print(sig.parse("IsDog(diff(x, Rex))"))   # terms can be nested

The parser also checks the formula against the declarations, before any data or network is involved. A name must be declared:

In [ ]:
try:
    sig.parse("Barks(x)")
except NameError as e:
    print("NameError:", e)

Each predicate and function must get the declared number of arguments, each of the declared sort. `embed(x)` is an `Embedding`, not an `Animal`:

In [ ]:
for formula in ["Similar(x)", "IsDog(embed(x))"]:
    try:
        sig.parse(formula)
    except TypeError as e:
        print("TypeError:", e)

## Part 2 — The Interpretation: giving the symbols a meaning

An `Interpretation` is built on a signature and grounds its symbols with `interp[name] = value`. It is an `nn.Module`: its networks and learned constants appear in `interp.parameters()`, and `interp.to(device)` moves them.

### 2.1 Sorts become types

A sort is grounded with a `Type`, which gives the **shape of one individual** and a name for its feature axis. Here an animal is a vector in ℝ² of two features, its size and its furriness, and an embedding a vector in ℝ⁸. `Bool` and `Real` already have their types (`sltn.Bool`, `sltn.Real`).

In [ ]:
interp = Interpretation(sig).to(device)

interp["Animal"] = Type("R2", shape=(2,))
interp["Embedding"] = Type("R8", shape=(8,))

print(sltn.Bool)
print(sltn.Real)

### 2.2 Constants

A constant is grounded with a tensor of **exactly the shape of its sort**, here `(2,)`, with no batch axis. Use an `nn.Parameter` for a constant whose features should be learned: it then shows up in `interp.parameters()`.

In [ ]:
interp["Fido"] = torch.tensor([2.1, 3.0])
interp["Rex"] = nn.Parameter(torch.zeros(2))

print([name for name, _ in interp.named_parameters()])

The shape is checked against the sort's `Type` when you ground the constant:

In [ ]:
try:
    interp["Fido"] = torch.tensor([2.1, 3.0, 0.5])   # an Animal has 2 features, not 3
except ValueError as e:
    print("ValueError:", e)

### 2.3 Variables

A variable is grounded with a batch of individuals: a tensor of shape `[n, *shape of the sort]`. Here `x` gets 10 animals and `y` gets 7.

In [ ]:
interp["x"] = torch.randn(10, 2)
interp["y"] = torch.randn(7, 2)

### 2.4 Predicates

A predicate is grounded with an `nn.Module` (trainable) or a plain callable (fixed). The rules for the grounding:

- it receives **one tensor per argument**, each of shape `[B, *shape of the argument's sort]`. `B` is a batch of inputs; Part 3 shows how sLTN builds it;
- it returns a tensor of shape `[B, 1]`: one truth degree per input, **with a final axis of size 1**, and values in **[0, 1]**.

In [ ]:
# A neural predicate: trainable
class IsDogNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, 16), nn.ELU(), nn.Linear(16, 1), nn.Sigmoid())

    def forward(self, a):                  # a: [B, 2]
        return self.net(a)                 # [B, 1]

interp["IsDog"] = IsDogNet()

# A fixed predicate: a plain callable, closer to mu means more true
mu = torch.tensor([0.5, 0.5], device=device)
interp["Near"] = lambda a: torch.exp(-torch.norm(a - mu, dim=-1, keepdim=True))

# A binary predicate: one tensor per argument
class SimilarNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 8), nn.ELU(), nn.Linear(8, 1), nn.Sigmoid())

    def forward(self, a, b):               # a, b: [B, 2]
        return self.net(torch.cat([a, b], dim=-1))

interp["Similar"] = SimilarNet()

`interp.to(device)` was called before these groundings were added: the networks are moved to that device when they are grounded. A tensor captured by a lambda, like `mu`, is not a grounding, so it was created on `device` by hand. The same goes for a network used inside a lambda: it is not registered, so it is not moved by `.to()`, not listed by `interp.parameters()`, and never trained. A trainable grounding must be an `nn.Module` (or an `nn.Parameter`): wrap the network in a small module, as `IsDogNet` does.

Training catches this mistake: `kb_backward` (Tutorial 4) raises an error when the knowledge base depends on trainable parameters that are neither in the optimizer nor in the interpretation. We try it on a copy (see 3.5), so that `interp` stays correct:

In [ ]:
bad = interp.copy()
net = nn.Sequential(nn.Linear(2, 1), nn.Sigmoid()).to(device)
bad["Near"] = lambda a: net(a)                  # a network inside a lambda: not registered
opt = torch.optim.Adam(bad.parameters())        # so net is not in the optimizer

try:
    sltn.kb_backward(opt, bad, {"near": "Near(Fido)"})  # kb_backward will be explained in Tutorial 4
except ValueError as e:
    print("ValueError:", e)

### 2.5 Functions

A function is grounded the same way. It returns `[B, *shape of the output sort]`: here `embed` returns `[B, 8]` and `diff` returns `[B, 2]`.

In [ ]:
interp["embed"] = nn.Linear(2, 8)
interp["diff"] = lambda a, b: a - b

print(interp)

### 2.6 Two common mistakes

A grounding that breaks the rules of 2.4 is caught only when a formula is evaluated, and the message is about tensors rather than about your predicate. We try two mistakes on a copy, so that `interp` stays correct.

**Forgetting the final axis of size 1.** The norm below drops the last axis (`keepdim` is missing), so the predicate returns `[B]` instead of `[B, 1]`:

In [ ]:
bad = interp.copy()
bad["Near"] = lambda a: torch.exp(-torch.norm(a - mu, dim=-1))   # no keepdim=True

try:
    bad("Near(x)")
except ValueError as e:
    print("ValueError:", e)

The error names the predicate, the shape it returned and the shape it should have returned.

**Returning values outside [0, 1].** A truth degree must lie in [0, 1]; a network without a final sigmoid usually breaks this:

In [ ]:
bad["Near"] = lambda a: torch.norm(a - mu, dim=-1, keepdim=True)   # a distance, not a truth degree

try:
    bad("Near(x)")
except ValueError as e:
    print("ValueError:", e)

## Part 3 — Evaluation

### 3.1 The result is a tensor with named axes

Calling the interpretation on a formula (a parsed one, or a string) evaluates it. The result is an `sltn.Tensor`: a PyTorch tensor (`.value`) whose axes have **names** and **roles**:

- a **variable** axis for each free variable of the formula: one entry per individual;
- **domain** axes for the features of one individual. For a formula, this is the `bool` axis of size 1 that holds the truth degree.

(Tutorial 5 introduces the third role, **structural** axes, for time steps or positions in a sequence.)

In [ ]:
result = interp("IsDog(x)")
print(result)
print(result.value.shape)

`x` was grounded as `[10, 2]`: a variable axis `x` of size 10 and a domain axis `r2` of size 2. `IsDog` replaced the 2 features of each animal with one truth degree, so the result is `[x: 10, bool: 1]`: how much each of the 10 animals is a dog.

### 3.2 Several variables: every combination

With two variables, the result has one axis per variable, and the predicate is evaluated on **every pair**: `Similar(x, y)` calls `SimilarNet` on 10 × 7 = 70 pairs. A constant has no variable axis, so it adds none.

In [ ]:
for formula in ["Similar(x, y)", "Similar(x, Fido)", "Similar(Fido, Rex)"]:
    r = interp(formula)
    print(f"{formula:20s} axes {[a.name for a in r.axes]}   shape {tuple(r.value.shape)}")

This is the semantics of LTN: a formula with free variables is evaluated at every combination of their individuals, and `B` in 2.4 is that number of combinations. It also means the cost grows with the **product** of the batch sizes.

### 3.3 Terms

Terms can be evaluated too. A constant has only its domain axis; a function's result has the domain axis of its output sort (`r8` for `embed`); terms nest inside predicates.

In [ ]:
print(interp("Fido"))
print(interp("embed(x)"))
print(interp("Near(diff(x, Rex))"))

Reading a symbol with `interp[name]` returns what you grounded it with (a tensor, a module, a callable). Calling `interp(...)` evaluates it and returns an `sltn.Tensor`:

In [ ]:
print(type(interp["x"]), tuple(interp["x"].shape))
print(interp("x"))

### 3.4 The same formula on other data

The formula doesn't change when the data does: ground the variable with another batch and evaluate again.

In [ ]:
phi = sig.parse("Near(x)")
print(interp(phi).value.shape)

interp["x"] = torch.randn(5, 2)    # a new batch of 5 animals
print(interp(phi).value.shape)

### 3.5 The same formula with other groundings

The formula doesn't change when the meaning does either. `copy()` returns an interpretation that shares the groundings; replacing one in the copy leaves the original untouched. (`interp.with_updates(Near=...)` does the same in one call.)

In [ ]:
wide = interp.copy()
wide["Near"] = lambda a: torch.exp(-0.1 * torch.norm(a - mu, dim=-1, keepdim=True))   # a wider notion of "near"

print("interp:", interp(phi).value.flatten())
print("wide  :", wide(phi).value.flatten())

## Part 4 — Reading groundings back, and devices

`interp[name]` returns the grounding, which is useful to inspect a trained network, a learned constant or a batch:

In [ ]:
print("IsDog :", interp["IsDog"])
print("Rex   :", interp["Rex"])                       # an nn.Parameter: learned by an optimiser
print("x     :", tuple(interp["x"].shape), interp["x"].device)

print("devices:", interp.device, next(interp["IsDog"].parameters()).device, next(interp["embed"].parameters()).device)

After `interp.to(device)`, the tensors and networks you ground are moved to that device. Move the interpretation itself with `interp.to(...)`, and create tensors captured by lambdas (like `mu`) on the right device yourself. Put trainable networks in modules, not in lambdas.

## Summary

- A `Signature` declares sorts, constants, variables, predicates and functions. Formulas are parsed and checked against it (names, number and sorts of arguments), with no values.
- An `Interpretation` grounds every symbol: sorts with a `Type`, individuals with tensors of the sort's shape, predicates and functions with modules or callables.
- A predicate's grounding receives `[B, *sort shape]` per argument and returns `[B, 1]` with values in [0, 1].
- Evaluating gives an `sltn.Tensor` with one variable axis per free variable: every combination of individuals is evaluated.
- The same formula can be evaluated on other data, or with other groundings, without being changed.

---

**Next:** [Tutorial 2 — FOL Formulas](2-FOLFormulas.ipynb) shows how to combine predicates into full formulas with connectives and quantifiers.